# Day 4 - Fund Performance Analytics

This notebook computes return, risk and benchmark metrics for the Bluestock mutual fund dataset. I kept the calculations in separate blocks so that it is easier to review and explain during evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import linregress

DATA = Path('../data/raw')
OUT = Path('../data/processed')
CHARTS = Path('../reports/charts')
OUT.mkdir(parents=True, exist_ok=True)
CHARTS.mkdir(parents=True, exist_ok=True)

fund_master = pd.read_csv(DATA / '01_fund_master.csv')
nav = pd.read_csv(DATA / '02_nav_history.csv')
bench = pd.read_csv(DATA / '10_benchmark_indices.csv')

fund_master['amfi_code'] = fund_master['amfi_code'].astype(str)
nav['amfi_code'] = nav['amfi_code'].astype(str)
nav['date'] = pd.to_datetime(nav['date'])
bench['date'] = pd.to_datetime(bench['date'])

nav = nav.drop_duplicates(['amfi_code', 'date']).sort_values(['amfi_code', 'date'])
nav = nav[nav['nav'] > 0].copy()
nav.head()

## 1. Daily returns

Daily return is calculated scheme-wise using NAV percentage change. The first row for every scheme is expected to be blank because there is no previous NAV value for comparison.

In [ ]:
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
print(nav['daily_return'].describe())

nav.to_csv(OUT / 'returns_computed.csv', index=False)

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(nav['daily_return'].dropna().clip(-0.06, 0.06), bins=80)
plt.title('Daily Return Distribution Across Fund NAV History')
plt.xlabel('Daily return')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(CHARTS / 'daily_return_distribution.png', dpi=150)
plt.show()

The return distribution is mostly centred around zero, which is expected for daily NAV returns. A few larger positive and negative days are present, but they are not dominating the overall distribution.

In [ ]:
bench = bench.sort_values(['index_name', 'date'])
bench['benchmark_return'] = bench.groupby('index_name')['close_value'].pct_change()
nifty100 = bench[bench['index_name'] == 'NIFTY100'][['date', 'benchmark_return']].dropna()
nifty100 = nifty100.rename(columns={'benchmark_return': 'nifty100_return'})

## 2. CAGR, Sharpe, Sortino, Alpha, Beta and Drawdown

Risk-free rate is taken as 6.5% annually. Alpha and beta are calculated using daily fund returns against NIFTY 100 daily returns.

In [ ]:
def cagr_for_group(g, years):
    end_date = g['date'].max()
    start_cut = end_date - pd.DateOffset(years=years)
    hist = g[g['date'] >= start_cut].sort_values('date')
    if len(hist) < 2:
        return np.nan
    start_nav = hist.iloc[0]['nav']
    end_nav = hist.iloc[-1]['nav']
    actual_years = (hist.iloc[-1]['date'] - hist.iloc[0]['date']).days / 365.25
    if actual_years <= 0 or start_nav <= 0:
        return np.nan
    return (end_nav / start_nav) ** (1 / actual_years) - 1

def max_dd_info(g):
    g = g.sort_values('date').copy()
    running_max = g['nav'].cummax()
    drawdown = g['nav'] / running_max - 1
    trough_idx = drawdown.idxmin()
    peak_nav = running_max.loc[trough_idx]
    peak_rows = g.loc[:trough_idx][g.loc[:trough_idx, 'nav'] == peak_nav]
    peak_date = peak_rows.iloc[-1]['date'] if len(peak_rows) else pd.NaT
    trough_date = g.loc[trough_idx, 'date']
    return drawdown.min(), peak_date, trough_date

In [ ]:
rf = 0.065
records = []

for code, g in nav.groupby('amfi_code'):
    g = g.sort_values('date')
    r = g['daily_return'].dropna()
    ann_return = (1 + r.mean()) ** 252 - 1
    ann_std = r.std() * np.sqrt(252)
    sharpe = (ann_return - rf) / ann_std if ann_std > 0 else np.nan
    downside = r[r < 0]
    downside_std = downside.std() * np.sqrt(252)
    sortino = (ann_return - rf) / downside_std if downside_std > 0 else np.nan

    max_dd, peak_date, trough_date = max_dd_info(g)

    merged = g[['date', 'daily_return']].dropna().merge(nifty100, on='date', how='inner')
    if len(merged) > 20:
        lr = linregress(merged['nifty100_return'], merged['daily_return'])
        alpha = lr.intercept * 252
        beta = lr.slope
        r_squared = lr.rvalue ** 2
    else:
        alpha, beta, r_squared = np.nan, np.nan, np.nan

    records.append({
        'amfi_code': code,
        'return_1yr_pct': cagr_for_group(g, 1) * 100,
        'return_3yr_pct': cagr_for_group(g, 3) * 100,
        'return_5yr_pct': cagr_for_group(g, 5) * 100,
        'annualised_return_pct': ann_return * 100,
        'annualised_volatility_pct': ann_std * 100,
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'alpha': alpha,
        'beta': beta,
        'r_squared': r_squared,
        'max_drawdown_pct': max_dd * 100,
        'drawdown_start': peak_date,
        'drawdown_end': trough_date
    })

metrics = pd.DataFrame(records)
metrics = metrics.merge(
    fund_master[['amfi_code','scheme_name','fund_house','category','sub_category','expense_ratio_pct','benchmark','risk_category']],
    on='amfi_code', how='left'
)
metrics.head()

In [ ]:
metrics.sort_values('sharpe_ratio', ascending=False).head(10)[
    ['scheme_name','fund_house','return_3yr_pct','sharpe_ratio','sortino_ratio','alpha','beta','max_drawdown_pct']
]

In [ ]:
metrics.to_csv(OUT / 'performance_metrics.csv', index=False)
metrics[['amfi_code','scheme_name','fund_house','alpha','beta','r_squared','benchmark']].sort_values('alpha', ascending=False).to_csv(OUT / 'alpha_beta.csv', index=False)
metrics[['amfi_code','scheme_name','return_1yr_pct','return_3yr_pct','return_5yr_pct']].sort_values('return_3yr_pct', ascending=False).to_csv(OUT / 'cagr_report.csv', index=False)

## 3. Fund scorecard

The composite score gives higher weight to three-year return and Sharpe ratio, while still considering alpha, expense ratio and drawdown control.

In [ ]:
def pct_rank(s, ascending=True):
    return s.rank(pct=True, ascending=ascending) * 100

score = metrics.copy()
score['score_3yr_return'] = pct_rank(score['return_3yr_pct'], ascending=True)
score['score_sharpe'] = pct_rank(score['sharpe_ratio'], ascending=True)
score['score_alpha'] = pct_rank(score['alpha'], ascending=True)
score['score_expense'] = pct_rank(score['expense_ratio_pct'], ascending=False)
score['score_drawdown'] = pct_rank(score['max_drawdown_pct'], ascending=True)

score['fund_score'] = (
    0.30 * score['score_3yr_return'] +
    0.25 * score['score_sharpe'] +
    0.20 * score['score_alpha'] +
    0.15 * score['score_expense'] +
    0.10 * score['score_drawdown']
).round(2)

fund_scorecard = score[[
    'amfi_code','scheme_name','fund_house','category','sub_category','return_3yr_pct',
    'sharpe_ratio','alpha','expense_ratio_pct','max_drawdown_pct','fund_score'
]].sort_values('fund_score', ascending=False)

fund_scorecard.to_csv(OUT / 'fund_scorecard.csv', index=False)
fund_scorecard.head(10)

## 4. Benchmark comparison and tracking error

For the chart, the top five scorecard funds are indexed to 100 and compared with NIFTY 50 and NIFTY 100 over the latest three-year period.

In [ ]:
end = nav['date'].max()
start = end - pd.DateOffset(years=3)
top5_codes = fund_scorecard.head(5)['amfi_code'].tolist()
plot_nav = nav[(nav['amfi_code'].isin(top5_codes)) & (nav['date'] >= start)]
plot_nav = plot_nav.merge(fund_master[['amfi_code','scheme_name']], on='amfi_code', how='left')
bench_subset = bench[(bench['index_name'].isin(['NIFTY50','NIFTY100'])) & (bench['date'] >= start)]

plt.figure(figsize=(13,7))
for code, g in plot_nav.groupby('amfi_code'):
    g = g.sort_values('date')
    indexed = g['nav'] / g['nav'].iloc[0] * 100
    plt.plot(g['date'], indexed, linewidth=1.4, label=g['scheme_name'].iloc[0][:35])

for idx_name, g in bench_subset.groupby('index_name'):
    g = g.sort_values('date')
    indexed = g['close_value'] / g['close_value'].iloc[0] * 100
    plt.plot(g['date'], indexed, linestyle='--', linewidth=1.8, label=idx_name)

plt.title('Top 5 Fund Scorecard Funds vs NIFTY 50 and NIFTY 100')
plt.xlabel('Date')
plt.ylabel('Indexed value')
plt.legend(fontsize=8, ncol=2)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS / 'benchmark_comparison_top5.png', dpi=160)
plt.show()

In [ ]:
tracking_rows = []
bench_returns = bench[bench['index_name'].isin(['NIFTY50','NIFTY100'])][['date','index_name','benchmark_return']].dropna()

for code in top5_codes:
    fund_returns = nav[nav['amfi_code'] == code][['date','daily_return']].dropna()
    fund_name = fund_master.loc[fund_master['amfi_code'] == code, 'scheme_name'].iloc[0]
    for idx_name in ['NIFTY50','NIFTY100']:
        br = bench_returns[bench_returns['index_name'] == idx_name][['date','benchmark_return']]
        merged = fund_returns.merge(br, on='date', how='inner')
        merged = merged[merged['date'] >= start]
        tracking_error = (merged['daily_return'] - merged['benchmark_return']).std() * np.sqrt(252) * 100
        tracking_rows.append({
            'amfi_code': code,
            'scheme_name': fund_name,
            'benchmark': idx_name,
            'tracking_error_pct': tracking_error,
            'observations': len(merged)
        })

tracking_error = pd.DataFrame(tracking_rows)
tracking_error.to_csv(OUT / 'tracking_error.csv', index=False)
tracking_error

## Notes / Findings

1. Daily returns are centred close to zero, which is reasonable for daily NAV movement.
2. The scorecard is more balanced than ranking only by returns because it includes risk, alpha, expense ratio and drawdown.
3. Sharpe ratio helps identify funds where the return is better adjusted for volatility.
4. Sortino ratio is useful here because it focuses only on negative-return days.
5. Beta values close to 1 suggest funds moving broadly in line with NIFTY 100.
6. Positive alpha suggests outperformance after accounting for market movement.
7. Maximum drawdown highlights the worst peak-to-trough fall and is important for risk discussion.
8. The benchmark comparison chart uses indexed values, so the chart focuses on relative growth instead of absolute NAV levels.
9. Tracking error shows how closely or loosely the top scorecard funds moved versus NIFTY 50 and NIFTY 100.
10. The final scorecard should not be read as investment advice; it is a project ranking model based on selected metrics.